# Hotel Booking Cancellation Prediction — Sprint 3
## Optimization & Final Model

This notebook implements the six Sprint 3 stages using `hotel_bookings.csv`:

1. Feature engineering
2. Feature selection
3. Hyperparameter tuning
4. Final model training
5. Evaluation on unseen test data
6. Model serialization (`.joblib`)

**Target:** `is_canceled` (1 = canceled, 0 = not canceled).

> Place `hotel_bookings.csv` in the same folder as this notebook, or update `DATA_PATH` below. The notebook uses a stratified train/test split and keeps the test set out of hyperparameter tuning.

In [ ]:
# If needed, install dependencies by uncommenting and running:
# %pip install pandas numpy scikit-learn joblib matplotlib seaborn

import warnings
warnings.filterwarnings("ignore")

import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    ConfusionMatrixDisplay, RocCurveDisplay
)

RANDOM_STATE = 42
TARGET = "is_canceled"
DATA_PATH = "hotel_bookings.csv"  # Change this path if your CSV is elsewhere


In [ ]:
# 1. Load dataset
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df[TARGET].value_counts(dropna=False).rename("count").to_frame())


## 1. Feature Engineering

Create booking-level aggregates, ratios and seasonal features. The columns `reservation_status` and `reservation_status_date` are excluded because they can reveal the booking's eventual outcome. `assigned_room_type` is also excluded to reduce the risk of using information determined after booking.

In [ ]:
def engineer_features(data: pd.DataFrame) -> pd.DataFrame:
    x = data.copy()

    # Exclude post-outcome / operational fields to avoid target leakage.
    x = x.drop(
        columns=["reservation_status", "reservation_status_date",
                 "assigned_room_type"],
        errors="ignore"
    )

    x["total_guests"] = x[["adults", "children", "babies"]].sum(axis=1)
    x["total_nights"] = (
        x["stays_in_weekend_nights"] + x["stays_in_week_nights"]
    )
    x["weekend_night_share"] = (
        x["stays_in_weekend_nights"] / (x["total_nights"] + 1)
    )
    x["previous_booking_count"] = (
        x["previous_cancellations"] +
        x["previous_bookings_not_canceled"]
    )
    x["previous_cancel_rate"] = (
        x["previous_cancellations"] /
        (x["previous_booking_count"] + 1)
    )
    x["special_requests_per_guest"] = (
        x["total_of_special_requests"] / (x["total_guests"] + 1)
    )
    x["adr_per_guest"] = (
        x["adr"] / x["total_guests"].clip(lower=1)
    )

    month_num = pd.to_datetime(
        x["arrival_date_month"], format="%B"
    ).dt.month
    x["arrival_month_sin"] = np.sin(2 * np.pi * month_num / 12)
    x["arrival_month_cos"] = np.cos(2 * np.pi * month_num / 12)
    x = x.drop(columns=["arrival_date_month"])

    return x


y = df[TARGET].astype(int)
X = engineer_features(df.drop(columns=[TARGET]))

print("Feature matrix shape after engineering:", X.shape)
display(X.head())


## 2. Train/test split and feature selection

Use a stratified 80/20 split. The correlation filter is fitted using **training data only** and removes one feature from numeric pairs with absolute correlation greater than 0.95. Categorical columns are one-hot encoded inside the model pipeline.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)

numeric_all = X_train.select_dtypes(include=np.number).columns.tolist()
corr = X_train[numeric_all].corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))

correlated_to_drop = [
    col for col in upper.columns if any(upper[col] > 0.95)
]
numeric_features = [c for c in numeric_all if c not in correlated_to_drop]
categorical_features = X_train.select_dtypes(exclude=np.number).columns.tolist()

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Numeric features removed for high correlation:", correlated_to_drop)
print("Numeric features retained:", len(numeric_features))
print("Categorical features:", len(categorical_features))


In [ ]:
# Visualize numeric correlations (training partition only)
plt.figure(figsize=(14, 10))
sns.heatmap(
    X_train[numeric_all].corr(),
    cmap="coolwarm",
    center=0,
    annot=False
)
plt.title("Training Data — Numeric Feature Correlation")
plt.tight_layout()
plt.show()


## 3. Hyperparameter Tuning

The preprocessing and estimator are combined in one pipeline to prevent preprocessing leakage across cross-validation folds. A small stratified tuning subset and compact randomized search keep runtime manageable. Increase `n_iter` and the parameter ranges if you have more compute time.

In [ ]:
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=100,
        sparse_output=False
    ))
])

preprocessor = ColumnTransformer([
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

pipeline = Pipeline([
    ("prep", preprocessor),
    ("model", HistGradientBoostingClassifier(
        random_state=RANDOM_STATE,
        early_stopping=True
    ))
])

param_distributions = {
    "model__learning_rate": [0.05, 0.08, 0.1],
    "model__max_iter": [150, 220],
    "model__max_leaf_nodes": [15, 31],
    "model__l2_regularization": [0.0, 1.0]
}

# Stratified tuning subset: up to 12,500 rows from each class.
tune_idx = y_train.groupby(y_train, group_keys=False).apply(
    lambda s: s.sample(n=min(12500, len(s)), random_state=RANDOM_STATE)
).index

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=2,
    scoring="roc_auc",
    cv=2,
    random_state=RANDOM_STATE,
    n_jobs=1,
    verbose=1
)

search.fit(X_train.loc[tune_idx], y_train.loc[tune_idx])

print("Best CV ROC-AUC:", search.best_score_)
print("Best parameters:")
print(search.best_params_)


## 4. Build the final model

Refit the selected pipeline on the **entire training partition** using the best hyperparameters. The test partition remains untouched until final evaluation.

In [ ]:
final_model = search.best_estimator_
final_model.fit(X_train, y_train)

print("Final model fitted on all training rows.")


## 5. Final Evaluation on Unseen Test Data

Report accuracy, precision, recall, F1 and ROC-AUC. The confusion matrix uses the order `[[TN, FP], [FN, TP]]`.

In [ ]:
y_pred = final_model.predict(X_test)
y_prob = final_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob)),
    "confusion_matrix": confusion_matrix(y_test, y_pred).tolist(),
    "best_params": search.best_params_,
    "cv_roc_auc": float(search.best_score_),
    "correlated_features_removed": correlated_to_drop
}

display(pd.DataFrame({
    "Metric": ["Accuracy", "Precision", "Recall", "F1 score", "ROC-AUC"],
    "Score": [
        metrics["accuracy"], metrics["precision"], metrics["recall"],
        metrics["f1"], metrics["roc_auc"]
    ]
}))

print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Not canceled", "Canceled"]))


In [ ]:
ConfusionMatrixDisplay.from_predictions(
    y_test, y_pred, display_labels=["Not canceled", "Canceled"],
    cmap="Blues", values_format="d"
)
plt.title("Confusion Matrix — Test Set")
plt.tight_layout()
plt.show()

RocCurveDisplay.from_predictions(y_test, y_prob)
plt.title("ROC Curve — Test Set")
plt.tight_layout()
plt.show()


### Permutation feature importance

Permutation importance measures how much the fitted model's ROC-AUC changes when one input column is shuffled. This is an interpretation step on a sample of the held-out test data; do not use it to repeatedly tune the model against the test set.

In [ ]:
from sklearn.inspection import permutation_importance

importance_sample = min(3000, len(X_test))
X_imp = X_test.sample(n=importance_sample, random_state=RANDOM_STATE)
y_imp = y_test.loc[X_imp.index]

perm = permutation_importance(
    final_model, X_imp, y_imp,
    scoring="roc_auc",
    n_repeats=3,
    random_state=RANDOM_STATE,
    n_jobs=1
)

importance = pd.Series(
    perm.importances_mean, index=X_imp.columns
).sort_values(ascending=False)

display(importance.head(15).rename("mean ROC-AUC decrease").to_frame())

importance.head(15).sort_values().plot(kind="barh", figsize=(9, 6))
plt.title("Top Permutation Feature Importances")
plt.xlabel("Mean decrease in ROC-AUC")
plt.tight_layout()
plt.show()


## 6. Model Serialization

Save the complete fitted pipeline (preprocessing + classifier), plus the metrics. For new bookings, call `engineer_features()` first, then pass the resulting DataFrame to the saved pipeline. Do not include the target or excluded post-outcome columns.

In [ ]:
MODEL_PATH = "hotel_cancellation_final_model.joblib"
METRICS_PATH = "hotel_model_metrics.json"

joblib.dump(final_model, MODEL_PATH, compress=3)

# Store JSON-safe values
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)

print(f"Saved model: {MODEL_PATH}")
print(f"Saved metrics: {METRICS_PATH}")


## Example: Load model and predict new bookings

Replace the example with a DataFrame containing new booking records and the same raw input columns as the training CSV (excluding `is_canceled`). The same feature engineering function must be applied.

In [ ]:
# Example usage:
# loaded_model = joblib.load("hotel_cancellation_final_model.joblib")
# new_bookings = pd.read_csv("new_hotel_bookings.csv")
# new_features = engineer_features(new_bookings)
# cancellation_predictions = loaded_model.predict(new_features)
# cancellation_probabilities = loaded_model.predict_proba(new_features)[:, 1]
#
# output = new_bookings.copy()
# output["predicted_is_canceled"] = cancellation_predictions
# output["cancellation_probability"] = cancellation_probabilities
# display(output.head())


## Conclusion

Sprint 3 is complete when all cells run successfully and the `.joblib` model and metrics JSON are created. This evaluation is a random stratified split; for production readiness, additionally validate on a later time period and choose a decision threshold based on the hotel's costs of missed cancellations versus unnecessary cancellation flags.